In [ ]:
import random
import numpy as np
import os
import tensorflow as tf

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices("GPU"))

In [ ]:
import json
from PIL import Image
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

In [ ]:
import kagglehub

path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")

DATA_DIR = path
for root, dirs, files in os.walk(path):
    if "color" in root.lower() and len(dirs) > 5:
        DATA_DIR = root
        break

print("Dataset root directory:", DATA_DIR)

classes = sorted(os.listdir(DATA_DIR))
print(f"Total classes: {len(classes)}")
for idx, c in enumerate(classes):
    print(f"  [{idx:2d}] {c}")

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode="nearest",
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset="training",
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset="validation",
    class_mode="categorical",
    shuffle=False
)

print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")

In [ ]:
class_indices = {v: k for k, v in train_generator.class_indices.items()}
print("Class Indices:", json.dumps(class_indices, indent=2))

with open("class_indices.json", "w") as f:
    json.dump(class_indices, f, indent=4)
print("Successfully saved class_indices.json!")

In [ ]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(class_indices), activation="softmax")
])

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("--- Phase 1: Training Classification Head ---")
history_phase1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("--- Phase 2: Fine-Tuning Top 30 Layers ---")
history_phase2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

In [ ]:
print("Evaluating on validation set...")
val_loss, val_acc = model.evaluate(val_generator)
print(f"Final Validation Accuracy: {val_acc * 100:.2f}%")
print(f"Final Validation Loss: {val_loss:.4f}")

In [ ]:
acc = history_phase1.history["accuracy"] + history_phase2.history["accuracy"]
val_acc = history_phase1.history["val_accuracy"] + history_phase2.history["val_accuracy"]
loss = history_phase1.history["loss"] + history_phase2.history["loss"]
val_loss = history_phase1.history["val_loss"] + history_phase2.history["val_loss"]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label="Training Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.axvline(x=4.5, color="r", linestyle="--", label="Fine-Tuning Start")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
plt.axvline(x=4.5, color="r", linestyle="--", label="Fine-Tuning Start")
plt.title("Training and Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def load_and_preprocess_image(image_path):
    img = Image.open(image_path)
    if img.mode != "RGB":
        img = img.convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img, dtype=np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

def predict_disease(model, image_path, class_indices):
    preprocessed_img = load_and_preprocess_image(image_path)
    prediction = model.predict(preprocessed_img, verbose=0)[0]
    predicted_class_index = int(np.argmax(prediction))
    predicted_class = class_indices.get(predicted_class_index, f"Class {predicted_class_index}")
    confidence = float(prediction[predicted_class_index]) * 100
    return predicted_class, confidence

print("Inference function defined successfully!")

In [ ]:
model.save("plant_disease_model.h5", include_optimizer=False)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("disease_model.tflite", "wb") as f:
    f.write(tflite_model)

print("Saved plant_disease_model.h5 and disease_model.tflite")